# ⚡ BƯỚC 3: SERVE MODEL VỚI OPENAI-COMPATIBLE API & CLOUDFLARE TUNNEL (Chạy trên Colab T4 16GB)
Notebook này phục vụ cho việc **Coding hàng ngày**. Bạn chỉ cần mở notebook này lên trên GPU T4 miễn phí:
1. Chọn 1 Model bạn muốn nạp (từ Hugging Face, bản đã Uncensor ở Bước 1, hoặc bản đã Fine-tune ở Bước 2).
2. Server khởi động API chuẩn OpenAI (`/v1/chat/completions`) có hỗ trợ Streaming Token.
3. Cloudflare Tunnel tự động tạo **đường link HTTPS công khai** (không cần token, không giới hạn băng thông).
4. Nhập đường link này vào **DeepSeek Harness** trên máy tính của bạn để bắt đầu lập trình!

In [ ]:
# @title 1. Cài đặt Server Dependencies trên Colab T4
import torch
!nvidia-smi

!pip install -q fastapi uvicorn transformers accelerate bitsandbytes torch pydantic requests

In [ ]:
# @title 2. Kết nối Google Drive (Nếu nạp model đã Uncensor hoặc Fine-tune)
from google.colab import drive
import os

drive.mount('/content/drive')
UNCENSORED_DIR = "/content/drive/MyDrive/ai_coding_models_uncensored"
FINETUNED_DIR = "/content/drive/MyDrive/ai_coding_models_finetuned"
print("✅ Google Drive đã kết nối thành công!")

In [ ]:
# @title 3. Chọn Nguồn & Model để nạp vào VRAM T4
# @markdown Chọn nguồn model bạn muốn chạy:
MODEL_SOURCE = "From Hugging Face" # @param ["From Hugging Face", "From Google Drive (Uncensored - Step 1)", "From Google Drive (Fine-tuned - Step 2)"]
HF_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct" # @param ["Qwen/Qwen2.5-Coder-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct", "Qwen/Qwen2.5-Coder-14B-Instruct"]
DRIVE_UNCENSORED_NAME = "Qwen2.5-Coder-7B-Heretic-Uncensored" # @param {type:"string"}
DRIVE_FINETUNED_NAME = "Qwen2.5-Coder-7B-Instruct-Finetuned-Final" # @param {type:"string"}
QUANTIZATION_4BIT = True # @param {type:"boolean"}

if MODEL_SOURCE == "From Google Drive (Uncensored - Step 1)":
    target_path = os.path.join(UNCENSORED_DIR, DRIVE_UNCENSORED_NAME)
    SELECTED_MODEL = target_path if os.path.exists(target_path) else HF_MODEL_NAME
elif MODEL_SOURCE == "From Google Drive (Fine-tuned - Step 2)":
    target_path = os.path.join(FINETUNED_DIR, DRIVE_FINETUNED_NAME)
    SELECTED_MODEL = target_path if os.path.exists(target_path) else HF_MODEL_NAME
else:
    SELECTED_MODEL = HF_MODEL_NAME

if "DeepSeek-Coder-V2-Lite" in SELECTED_MODEL:
    print("⚠️ Lưu ý: DeepSeek-Coder-V2-Lite là mô hình MoE 16B tham số. BẮT BUỘC dùng 4-bit Quantization trên GPU T4 16GB!")

print(f"🎯 Model được chọn để chạy trên GPU: {SELECTED_MODEL}")

In [ ]:
# @title 4. Tạo mã nguồn API Server chuẩn OpenAI
import os
os.makedirs("shared", exist_ok=True)

api_server_code = '''
import asyncio, json, time, uuid, torch, threading, uvicorn, argparse
from typing import Any, AsyncGenerator, Dict, List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer, BitsAndBytesConfig

app = FastAPI(title="Colab Backend", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

MODEL, TOKENIZER, MODEL_NAME = None, None, ""

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = "custom-model"
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.2
    top_p: Optional[float] = 0.95
    max_tokens: Optional[int] = 4096
    stream: Optional[bool] = False

def init_model(model_path_or_id: str, load_in_4bit: bool = True):
    global MODEL, TOKENIZER, MODEL_NAME
    print(f"🔄 Loading {model_path_or_id} (4bit={load_in_4bit})...")
    TOKENIZER = AutoTokenizer.from_pretrained(model_path_or_id, trust_remote_code=True)
    if TOKENIZER.pad_token is None: TOKENIZER.pad_token = TOKENIZER.eos_token
    TOKENIZER.padding_side = "left"
    kwargs = {"device_map": "auto", "trust_remote_code": True, "torch_dtype": torch.float16}
    if load_in_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True
        )
    MODEL = AutoModelForCausalLM.from_pretrained(model_path_or_id, **kwargs)
    MODEL_NAME = model_path_or_id
    print(f"✅ Loaded {model_path_or_id} successfully!")

@app.get("/health")
@app.get("/v1/health")
async def health():
    if MODEL is None or TOKENIZER is None:
        raise HTTPException(status_code=503, detail="Model is still loading...")
    return {"status": "ok", "model": MODEL_NAME}

@app.get("/v1/models")
async def list_models():
    return {"object": "list", "data": [{"id": MODEL_NAME, "object": "model", "created": int(time.time()), "owned_by": "colab"}]}

@app.post("/v1/chat/completions")
async def chat_completions(req: ChatCompletionRequest):
    global MODEL, TOKENIZER, MODEL_NAME
    if MODEL is None: raise HTTPException(503, "Model not ready")
    messages_payload = [{"role": m.role, "content": m.content} for m in req.messages]
    try: prompt_text = TOKENIZER.apply_chat_template(messages_payload, tokenize=False, add_generation_prompt=True)
    except: prompt_text = "".join([f"<|im_start|>{m.role}\n{m.content}<|im_end|>\n" for m in req.messages]) + "<|im_start|>assistant\n"
    inputs = TOKENIZER([prompt_text], return_tensors="pt").to(MODEL.device)
    cid = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    gen_kwargs = {
        **inputs,
        "max_new_tokens": req.max_tokens or 4096,
        "do_sample": req.temperature > 0 if req.temperature is not None else False,
        "temperature": req.temperature if req.temperature and req.temperature > 0 else None,
        "top_p": req.top_p if req.temperature and req.temperature > 0 else None,
        "pad_token_id": TOKENIZER.pad_token_id
    }
    if req.stream:
        async def sse_gen():
            streamer = TextIteratorStreamer(TOKENIZER, skip_prompt=True, skip_special_tokens=True)
            gen_kwargs["streamer"] = streamer
            threading.Thread(target=MODEL.generate, kwargs=gen_kwargs).start()
            yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': ''}, 'finish_reason': None}]})}\n\n"
            for new_text in streamer:
                if new_text:
                    yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {'content': new_text}, 'finish_reason': None}]})}\n\n"
                await asyncio.sleep(0.001)
            yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]})}\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(sse_gen(), media_type="text/event-stream")
    else:
        with torch.no_grad(): out = MODEL.generate(**gen_kwargs)
        in_len = inputs["input_ids"].shape[1]
        resp_text = TOKENIZER.decode(out[0][in_len:], skip_special_tokens=True)
        return {"id": cid, "object": "chat.completion", "created": int(time.time()), "model": MODEL_NAME, "choices": [{"index": 0, "message": {"role": "assistant", "content": resp_text}, "finish_reason": "stop"}]}
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, required=True)
    parser.add_argument("--port", type=int, default=8000)
    parser.add_argument("--4bit", dest="load_4bit", action="store_true", default=True)
    args = parser.parse_args()
    init_model(args.model, load_in_4bit=args.load_4bit)
    uvicorn.run(app, host="0.0.0.0", port=args.port)
'''
with open("shared/api_server.py", "w") as f: f.write(api_server_code)
print("✅ Code API Server đã cập nhật hoàn tất!")

In [ ]:
# @title 5. Khởi chạy Server & Mở Cloudflare Tunnel
import subprocess, time, urllib.request, re, os, requests

# 1. Chạy API Server ở background
cmd = ["python", "shared/api_server.py", "--model", SELECTED_MODEL, "--port", "8000"]
if QUANTIZATION_4BIT:
    cmd.append("--4bit")

server_log = open("server.log", "w")
server_proc = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT)
print("⏳ Đang nạp model vào GPU VRAM...")

# Health check loop chủ động thay vì sleep cố định
server_ready = False
for i in range(60):
    time.sleep(2)
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=1)
        if r.status_code == 200:
            server_ready = True
            print(f"✅ Server đã sẵn sàng sau {(i+1)*2} giây!")
            break
    except:
        pass

if not server_ready:
    print("⚠️ Server nạp hơi lâu, đang tiếp tục mở Tunnel...")

# 2. Tải cloudflared nếu chưa có
if not os.path.exists("./cloudflared"):
    print("📥 Đang tải cloudflared...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "./cloudflared")
    os.chmod("./cloudflared", 0o755)

# 3. Khởi chạy Cloudflare Tunnel
cf_log = open("cloudflared.log", "w")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=cf_log, stderr=subprocess.STDOUT)

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("cloudflared.log"):
        with open("cloudflared.log", "r") as f:
            matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if matches:
                tunnel_url = matches[0]
                break

if tunnel_url:
    print("\n" + "="*65)
    print("🎉 BACKEND SERVER ĐÃ KHỞI CHẠY THÀNH CÔNG!")
    print(f"👉 OPENAI BASE URL: {tunnel_url}/v1")
    print(f"👉 Dán link trên vào Base URL của DeepSeek Harness trên máy bạn!")
    print("="*65 + "\n")
else:
    print("❌ Chưa lấy được Tunnel URL. Vui lòng xem log: !cat cloudflared.log")